# 01 — Từ CSV đến Bronze

### 1. Xác định đường dẫn nguồn

Repo cung cấp `data/delivery_yt.csv`. Notebook và Prefect ưu tiên đối tượng
`s3a://lakehouse/delivery_yt.csv`. Khi thiếu đối tượng đó, hàm dùng file
`file:///home/jovyan/data/delivery_yt.csv` trong Jupyter. Hai biến
`SOURCE_PATH` và `LOCAL_SOURCE` cấu hình hai đường dẫn.

Chạy trong Jupyter Docker với MinIO sẵn sàng.

Cấu hình dùng chung khai báo địa chỉ Delta và các lớp Spark mà pipeline cần.
`get_spark()` lấy cấu hình MinIO và Delta từ `lakehouse_common.py`.

In [ ]:
"""Các bước xử lý Spark dùng chung cho notebook và flow Prefect."""
import os
import sys
from datetime import datetime
import uuid

sys.path.insert(0, os.getenv("NOTEBOOK_DIR", "/home/jovyan/work"))

import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor, LinearRegression
from pyspark.sql.window import Window

from lakehouse_common import (
    attach_dimension_keys, build_dimensions, courier_history, create_spark, haversine, join_courier_history,
    parse_timestamp, require_unique_orders, valid_coordinates,
)

BASE = "s3a://lakehouse/warehouse/"
BRONZE = BASE + "delivery_bronze"
SILVER = BASE + "delivery_silver"
REJECTED = BASE + "delivery_rejected"
DIM = BASE + "dim_courier_silver"
DIMS = {n: BASE + n + "_gold" for n in ("dim_region", "dim_aoi", "dim_date", "dim_hour")}
FACT = BASE + "fact_delivery_task_gold"
PRED = BASE + "eta_predictions_gold"
RUN_INFO = BASE + "pipeline_run_info"

NUM_FEATURES = ["distance_km", "accept_hour", "dow", "order_rank",
                "cum_avg_dist", "courier_train_avg_min"]
ETA_SAFE = BASE + "eta_safe_gold"
ETA_SAFE_METRICS = BASE + "eta_safe_metrics_gold"
ETA_PAPER = BASE + "eta_paper_gold"
ETA_PAPER_METRICS = BASE + "eta_paper_metrics_gold"


def get_spark(name="Delivery - Prefect pipeline"):
    return create_spark(name)

In [ ]:
spark = get_spark("Delivery - Bronze")

### 2. Đọc 17 cột, kiểm tra khóa và ghi Delta

`ingest_bronze()` đọc các cột dưới dạng chuỗi, kiểm tra `order_id` rồi
ghi `delivery_bronze`. Hàm đối soát khóa vừa ghi với CSV. Trường `source`
ở kết quả cho biết đường dẫn thực tế đã đọc.

In [ ]:
def _exists(spark, path):
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    return p.getFileSystem(spark._jsc.hadoopConfiguration()).exists(p)


def read_bronze_csv(spark, source):
    from src.eta.dataset import SOURCE_COLUMNS

    df = spark.read.csv(source, header=True, inferSchema=False, mode="FAILFAST")
    if df.columns != SOURCE_COLUMNS:
        raise ValueError(f"Sai cấu trúc CSV LaDe-D: {df.columns}")
    return df

In [ ]:
def reconcile_order_keys(source, output, label):
    """Check an exact one-to-one mapping, including missing and extra orders."""
    expected = source.select("order_id", F.lit(1).alias("expected"), F.lit(0).alias("actual"))
    actual = output.select("order_id", F.lit(0).alias("expected"), F.lit(1).alias("actual"))
    mismatches = (expected.unionByName(actual).groupBy("order_id").agg(
        F.sum("expected").alias("expected"), F.sum("actual").alias("actual"))
        .filter("order_id IS NULL OR trim(order_id) = '' OR expected <> 1 OR actual <> 1"))
    if mismatches.limit(1).count():
        raise ValueError(f"{label}: khóa order_id thiếu, dư hoặc trùng")

In [ ]:
def ingest_bronze(spark):
    src = os.getenv("SOURCE_PATH", "s3a://lakehouse/delivery_yt.csv")
    if not _exists(spark, src):
        src = os.getenv("LOCAL_SOURCE", "file:///home/jovyan/data/delivery_yt.csv")
    df = read_bronze_csv(spark, src)
    require_unique_orders(df)
    n = df.count()
    if n == 0:
        raise ValueError("CSV rỗng")
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(BRONZE))
    persisted = spark.read.format("delta").load(BRONZE)
    reconcile_order_keys(df, persisted, "CSV → Bronze")
    return {"source": src, "bronze_rows": persisted.count()}

In [1]:
result = ingest_bronze(spark)
result

{'source': 's3a://lakehouse/delivery_yt.csv', 'bronze_rows': 206431}

### 3. Xem dữ liệu đã ghi

Các cột thời gian ở Bronze giữ định dạng của CSV. Notebook 02 sẽ đổi kiểu.

In [ ]:
bronze = spark.read.format("delta").load(BRONZE)
print(BRONZE)
bronze.select("order_id", "courier_id", "aoi_id", "accept_time", "delivery_time").show(5, truncate=False)